# RNN MFCC-10
Trains multilabel and binary UUV BiLSTMs for normal, M-filtered, and W-filtered MFCC-10 data.

In [1]:
import sys
from pathlib import Path

PIPELINE_GITHUB_RAW_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/spectrogram_pipeline.py"  # Optional raw GitHub URL for spectrogram_pipeline.py
MODULE_FILE = "spectrogram_pipeline.py"
candidate_dirs = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/drive/MyDrive/STUDA/src")]
module_path = next((directory / MODULE_FILE for directory in candidate_dirs if (directory / MODULE_FILE).exists()), None)

if module_path is None and PIPELINE_GITHUB_RAW_URL:
    import urllib.request
    module_path = Path("/content") / MODULE_FILE
    urllib.request.urlretrieve(PIPELINE_GITHUB_RAW_URL, module_path)

if module_path is None or not module_path.exists():
    raise FileNotFoundError(f"{MODULE_FILE} was not found. Sync, upload, mount, or set PIPELINE_GITHUB_RAW_URL.")

sys.path.insert(0, str(module_path.parent))
print(f"Using pipeline module: {module_path}")


Using pipeline module: /content/spectrogram_pipeline.py


In [2]:
!pip install -q tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 765.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 111.8 MB/s eta 0:00:0000:0100:01


In [ ]:
import pandas as pd
import tensorflow as tf
from IPython.display import display
from google.colab import files, userdata

from spectrogram_pipeline import (
    build_rnn_models_for_variants, evaluate_models_for_variants, extract_zip,
    get_rnn_callbacks, plot_training_histories, prepare_mfcc_dataset_variants,
    save_artifacts, train_models_for_variants, zip_artifacts,
)

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU is attached. Select a GPU runtime and restart the session.")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"Using GPU: {gpus[0].name}")


IndentationError: expected an indented block after 'except' statement on line 19 (3283442088.py, line 20)

In [7]:
import os
import tensorflow as tf

print("COLAB_TPU_ADDR:", os.environ.get("COLAB_TPU_ADDR"))
print("TPU devices:", tf.config.list_logical_devices("TPU"))

COLAB_TPU_ADDR: None
TPU devices: []


In [ ]:
DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")
EPOCHS = 50
BATCH_SIZE = 64


In [6]:
kaggle_token = "KGAT_13e99a89ce99374acab577737d175634"

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / "access_token"
token_path.write_text(kaggle_token)
token_path.chmod(0o600)


In [ ]:
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
DATA_PATH = extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


In [ ]:
variants = prepare_mfcc_dataset_variants(DATA_PATH)
print("Normal train shape:", variants.normal.train_data.shape)
print("M train shape:", variants.m.train_data.shape)
print("W train shape:", variants.w.train_data.shape)


In [ ]:
multilabel_models = build_rnn_models_for_variants(variants, model_type="multilabel")

multilabel_histories = train_models_for_variants(
    multilabel_models, variants, model_type="multilabel", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_rnn_callbacks,
)


In [ ]:
binary_models = build_rnn_models_for_variants(variants, model_type="binary")

binary_histories = train_models_for_variants(
    binary_models, variants, model_type="binary", epochs=EPOCHS,
    batch_size=BATCH_SIZE, callback_factory=get_rnn_callbacks,
)


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])


In [ ]:
plot_training_histories(multilabel_histories, f"Multilabel RNN Training Curves - {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary RNN Training Curves - {DATASET_LABEL}")

save_dir = save_artifacts(
    f"/content/saved_artifacts/rnn_{DATASET_KEY}", DATASET_KEY, multilabel_models, binary_models,
    multilabel_histories, binary_histories, multilabel_results, binary_results,
)
comparison_results.to_csv(save_dir / f"rnn_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/rnn_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
